In [1]:
import pandas as pd
import numpy as np
import json
import os
from os.path import join
from tqdm import tqdm

In [2]:
folder = r'/home/ilya_treyvish/projects/lbnl_fdd/data/processed/SDAHU'

In [3]:
train_df = pd.read_csv(join(folder, 'train_df.csv'), index_col=[0, 1],
    parse_dates=[1])
train_target = pd.read_csv(join(folder, 'train_target.csv'), index_col=[0, 1],
    parse_dates=[1])
val_df = pd.read_csv(join(folder, 'val_df.csv'), index_col=[0, 1],
    parse_dates=[1])
val_target = pd.read_csv(join(folder, 'val_target.csv'), index_col=[0, 1],
    parse_dates=[1])
test_df = pd.read_csv(join(folder, 'test_df.csv'), index_col=[0, 1],
    parse_dates=[1])
test_target = pd.read_csv(join(folder, 'test_target.csv'), index_col=[0, 1],
    parse_dates=[1])

In [4]:
train_df.shape

(4448700, 25)

In [5]:
train_df.head()

CHWC_VLV_DM  ZONE_TEMP_4    SA_CFM  \
fault_type Datetime                                                  
0          2018-01-01 01:00:00          0.0    66.761345 -0.943331   
           2018-01-01 01:01:00          0.0    66.761290 -0.943167   
           2018-01-01 01:02:00          0.0    66.761185 -0.943027   
           2018-01-01 01:03:00          0.0    66.761020 -0.942867   
           2018-01-01 01:04:00          0.0    66.760740 -0.942719   

                                      SF_WAT    MA_TEMP  SYS_CTL  OA_DMPR_DM  \
fault_type Datetime                                                            
0          2018-01-01 01:00:00 -2.791519e-13  66.374680      0.0         0.0   
           2018-01-01 01:01:00 -2.790612e-13  66.374680      0.0         0.0   
           2018-01-01 01:02:00 -2.789696e-13  66.374680      0.0         0.0   
           2018-01-01 01:03:00 -2.788807e-13  66.374626      0.0         0.0   
           2018-01-01 01:04:00 -2.787895e-13  66.374626      0.0         0.0   

                                ZONE_TEMP_2   RA_TEMP      CHWC_VLV  ...  \
fault_type Datetime                                                  ...   
0          2018-01-01 01:00:00    66.767230  68.34175  2.635303e-21  ...   
           2018-01-01 01:01:00    66.766680  68.34786  2.479578e-21  ...   
           2018-01-01 01:02:00    66.766014  68.35378  2.380361e-21  ...   
           2018-01-01 01:03:00    66.765305  68.35948 -1.274259e-21  ...   
           2018-01-01 01:04:00    66.764530  68.36498  2.749987e-21  ...   

                                RA_DMPR_DM    SA_TEMP  RF_CS  RA_DMPR  \
fault_type Datetime                                                     
0          2018-01-01 01:00:00         1.0  66.374680    0.0      1.0   
           2018-01-01 01:01:00         1.0  66.374680    0.0      1.0   
           2018-01-01 01:02:00         1.0  66.374680    0.0      1.0   
           2018-01-01 01:03:00         1.0  66.374626    0.0      1.0   
           2018-01-01 01:04:00         1.0  66.374626    0.0      1.0   

                                  RA_CFM  RF_SPD_DM        RF_WAT  \
fault_type Datetime                                                 
0          2018-01-01 01:00:00  0.853194        0.0 -2.617164e-13   
           2018-01-01 01:01:00  0.853057        0.0 -2.616267e-13   
           2018-01-01 01:02:00  0.852918        0.0 -2.615462e-13   
           2018-01-01 01:03:00  0.852779        0.0 -2.614546e-13   
           2018-01-01 01:04:00  0.852641        0.0 -2.613691e-13   

                                ZONE_TEMP_1  ZONE_TEMP_3  SF_SPD_DM  
fault_type Datetime                                                  
0          2018-01-01 01:00:00     73.94652    67.027100        0.0  
           2018-01-01 01:01:00     73.97777    67.025894        0.0  
           2018-01-01 01:02:00     74.00836    67.024580        0.0  
           2018-01-01 01:03:00     74.03825    67.023150        0.0  
           2018-01-01 01:04:00     74.06741    67.021670        0.0  

[5 rows x 25 columns]

In [9]:
import gc
import time
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    log_loss
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")


# =========================================================
# 1. Вспомогательные функции
# =========================================================

def print_step(msg: str):
    print(f"\n{'=' * 80}\n{msg}\n{'=' * 80}")


def reduce_memory_df(df: pd.DataFrame, verbose: bool = False) -> pd.DataFrame:
    """
    Уменьшаем память:
    - float64 -> float32
    - int64 -> int32/int16/int8, если помещается
    """
    out = df.copy()

    if verbose:
        start_mem = out.memory_usage(deep=True).sum() / 1024**2
        print(f"Memory before: {start_mem:.2f} MB")

    for col in out.columns:
        col_dtype = out[col].dtype

        if pd.api.types.is_float_dtype(col_dtype):
            out[col] = out[col].astype(np.float32)

        elif pd.api.types.is_integer_dtype(col_dtype):
            cmin, cmax = out[col].min(), out[col].max()

            if cmin >= np.iinfo(np.int8).min and cmax <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif cmin >= np.iinfo(np.int16).min and cmax <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif cmin >= np.iinfo(np.int32).min and cmax <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)

    if verbose:
        end_mem = out.memory_usage(deep=True).sum() / 1024**2
        print(f"Memory after:  {end_mem:.2f} MB")
        print(f"Reduced by:    {100 * (start_mem - end_mem) / max(start_mem, 1e-9):.2f}%")

    return out


def extract_target(target_obj):
    """
    Универсально достаёт y из:
    - Series
    - DataFrame с колонкой fault_type
    - DataFrame из одной колонки
    - Index / np.ndarray / list
    """
    if isinstance(target_obj, pd.Series):
        return target_obj.values

    if isinstance(target_obj, pd.DataFrame):
        if "fault_type" in target_obj.columns:
            return target_obj["fault_type"].values
        elif target_obj.shape[1] == 1:
            return target_obj.iloc[:, 0].values
        else:
            raise ValueError(
                "target DataFrame has multiple columns and no 'fault_type' column"
            )

    if isinstance(target_obj, (pd.Index, np.ndarray, list)):
        return np.asarray(target_obj)

    raise TypeError(f"Unsupported target type: {type(target_obj)}")


def get_index_level_safe(df: pd.DataFrame, level_name: str):
    """
    Возвращает уровень индекса по имени, если это MultiIndex и такой уровень существует.
    Иначе None.
    """
    if isinstance(df.index, pd.MultiIndex):
        if level_name in df.index.names:
            return df.index.get_level_values(level_name)
    return None


def add_datetime_features_from_index(X: pd.DataFrame, dt_index_values) -> pd.DataFrame:
    """
    Добавляет простые признаки времени из уровня индекса Datetime.
    Использовать аккуратно: в симуляции это может дать leakage.
    """
    out = X.copy()
    dt = pd.to_datetime(dt_index_values)

    out["hour"] = dt.hour.astype(np.int8)
    out["minute"] = dt.minute.astype(np.int8)
    out["dayofweek"] = dt.dayofweek.astype(np.int8)
    out["day"] = dt.day.astype(np.int8)
    out["month"] = dt.month.astype(np.int8)

    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24).astype(np.float32)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24).astype(np.float32)
    out["minute_sin"] = np.sin(2 * np.pi * out["minute"] / 60).astype(np.float32)
    out["minute_cos"] = np.cos(2 * np.pi * out["minute"] / 60).astype(np.float32)

    return out


def prepare_features_from_multiindex(
    df: pd.DataFrame,
    use_datetime_features: bool = False,
    reduce_memory: bool = True,
    verbose: bool = False
) -> pd.DataFrame:
    """
    Подготовка X из DataFrame, где fault_type и Datetime сидят в индексе.
    Важно:
    - fault_type из индекса НЕ используем
    - Datetime можно либо игнорировать, либо извлечь как time-features
    """
    X = df.copy()

    # Если есть object-колонки, пробуем привести к numeric
    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            X[col] = pd.to_numeric(X[col], errors="coerce")

    # Добавить фичи из индекса Datetime, если нужно
    if use_datetime_features:
        dt_index = get_index_level_safe(X, "Datetime")
        if dt_index is not None:
            X = add_datetime_features_from_index(X, dt_index)
        elif "Datetime" in X.columns:
            X = X.copy()
            dt = pd.to_datetime(X["Datetime"])
            X["hour"] = dt.dt.hour.astype(np.int8)
            X["minute"] = dt.dt.minute.astype(np.int8)
            X["dayofweek"] = dt.dt.dayofweek.astype(np.int8)
            X["day"] = dt.dt.day.astype(np.int8)
            X["month"] = dt.dt.month.astype(np.int8)
            X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24).astype(np.float32)
            X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24).astype(np.float32)
            X["minute_sin"] = np.sin(2 * np.pi * X["minute"] / 60).astype(np.float32)
            X["minute_cos"] = np.cos(2 * np.pi * X["minute"] / 60).astype(np.float32)

    # Убираем потенциально опасные колонки/индекс
    if "fault_type" in X.columns:
        X = X.drop(columns=["fault_type"], errors="ignore")

    if "Datetime" in X.columns and not use_datetime_features:
        X = X.drop(columns=["Datetime"], errors="ignore")

    # Сам индекс модели не нужен
    X = X.reset_index(drop=True)

    # Все в float32
    X = X.astype(np.float32)

    if reduce_memory:
        X = reduce_memory_df(X, verbose=verbose)

    return X


def prepare_target(
    target_obj,
    label_encoder: Optional[LabelEncoder] = None
) -> Tuple[np.ndarray, LabelEncoder]:
    y_raw = extract_target(target_obj)

    if label_encoder is None:
        le = LabelEncoder()
        y = le.fit_transform(y_raw)
    else:
        le = label_encoder
        y = le.transform(y_raw)

    return y, le


def summarize_target(y: np.ndarray, label_encoder: Optional[LabelEncoder] = None, name: str = "target"):
    values, counts = np.unique(y, return_counts=True)
    print(f"{name}: n={len(y):,}, classes={len(values)}")
    if label_encoder is not None:
        decoded = label_encoder.inverse_transform(values)
        preview = list(zip(decoded[:10], counts[:10]))
        print(f"{name} preview counts: {preview}")
    else:
        preview = list(zip(values[:10], counts[:10]))
        print(f"{name} preview counts: {preview}")


def stratified_subsample(
    X: pd.DataFrame,
    y: np.ndarray,
    n_samples: int,
    random_state: int = 42
) -> Tuple[pd.DataFrame, np.ndarray]:
    """
    Стратифицированная подвыборка.
    Если n_samples >= len(X), возвращаем всё.
    """
    if n_samples >= len(X):
        return X.copy(), y.copy()

    rng = np.random.RandomState(random_state)
    classes, counts = np.unique(y, return_counts=True)
    n_classes = len(classes)

    base = n_samples // n_classes
    remainder = n_samples % n_classes

    idx_parts = []

    for i, cls in enumerate(classes):
        cls_idx = np.where(y == cls)[0]
        take = min(base + (1 if i < remainder else 0), len(cls_idx))
        chosen = rng.choice(cls_idx, size=take, replace=False)
        idx_parts.append(chosen)

    idx = np.concatenate(idx_parts)
    rng.shuffle(idx)

    return X.iloc[idx].copy(), y[idx].copy()


def batch_iterator(X: pd.DataFrame, y: np.ndarray, batch_size: int):
    n = len(X)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        yield start, end, X.iloc[start:end], y[start:end]


# =========================================================
# 2. Обёртки моделей
# =========================================================

class ScaledModelWrapper(BaseEstimator, ClassifierMixin):
    """
    Обёртка для модели, которую надо запускать через scaler.
    """
    def __init__(self, scaler, model):
        self.scaler = scaler
        self.model = model

    def predict(self, X):
        Xs = self.scaler.transform(X.values)
        return self.model.predict(Xs)

    def predict_proba(self, X):
        Xs = self.scaler.transform(X.values)
        return self.model.predict_proba(Xs)


# =========================================================
# 3. Метрики
# =========================================================

@dataclass
class ModelResult:
    model_name: str
    fit_seconds: float
    val_accuracy: float
    val_macro_f1: float
    val_weighted_f1: float
    val_logloss: Optional[float]
    notes: str


def evaluate_model(
    model,
    X_val: pd.DataFrame,
    y_val: np.ndarray,
    label_encoder: LabelEncoder,
    model_name: str,
    fit_seconds: float,
    notes: str = "",
    verbose: bool = True
):
    t0 = time.time()
    y_pred = model.predict(X_val)
    pred_seconds = time.time() - t0

    result = ModelResult(
        model_name=model_name,
        fit_seconds=fit_seconds,
        val_accuracy=accuracy_score(y_val, y_pred),
        val_macro_f1=f1_score(y_val, y_pred, average="macro"),
        val_weighted_f1=f1_score(y_val, y_pred, average="weighted"),
        val_logloss=None,
        notes=notes
    )

    if hasattr(model, "predict_proba"):
        try:
            y_proba = model.predict_proba(X_val)
            result.val_logloss = log_loss(
                y_val,
                y_proba,
                labels=np.arange(len(label_encoder.classes_))
            )
        except Exception:
            pass

    if verbose:
        print(f"{model_name}:")
        print(f"  fit_seconds    = {fit_seconds:.2f}")
        print(f"  pred_seconds   = {pred_seconds:.2f}")
        print(f"  val_accuracy   = {result.val_accuracy:.6f}")
        print(f"  val_macro_f1   = {result.val_macro_f1:.6f}")
        print(f"  val_weighted_f1= {result.val_weighted_f1:.6f}")
        if result.val_logloss is not None:
            print(f"  val_logloss    = {result.val_logloss:.6f}")
        if notes:
            print(f"  notes          = {notes}")

    return result, y_pred


def evaluate_on_test(
    model,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    label_encoder: LabelEncoder,
    model_name: str
) -> Dict[str, Any]:
    y_pred = model.predict(X_test)

    metrics = {
        "model_name": model_name,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_macro_f1": f1_score(y_test, y_pred, average="macro"),
        "test_weighted_f1": f1_score(y_test, y_pred, average="weighted"),
        "classification_report": classification_report(
            y_test,
            y_pred,
            target_names=[str(x) for x in label_encoder.classes_],
            digits=4
        ),
        "confusion_matrix": confusion_matrix(y_test, y_pred)
    }

    if hasattr(model, "predict_proba"):
        try:
            y_proba = model.predict_proba(X_test)
            metrics["test_logloss"] = log_loss(
                y_test,
                y_proba,
                labels=np.arange(len(label_encoder.classes_))
            )
        except Exception:
            metrics["test_logloss"] = None
    else:
        metrics["test_logloss"] = None

    return metrics


# =========================================================
# 4. Обучение моделей
# =========================================================

def train_dummy(X_train: pd.DataFrame, y_train: np.ndarray):
    model = DummyClassifier(strategy="prior")
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_seconds = time.time() - t0
    return model, fit_seconds


def train_sgd_incremental(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    classes: np.ndarray,
    batch_size: int = 200_000,
    alpha: float = 1e-4,
    random_state: int = 42,
    show_progress: bool = True
):
    """
    SGDClassifier(loss='log_loss') через partial_fit на полном train.
    """
    scaler = StandardScaler()
    clf = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=alpha,
        max_iter=1,
        tol=None,
        random_state=random_state
    )

    n_batches = int(np.ceil(len(X_train) / batch_size))
    t0 = time.time()

    # 1) partial_fit scaler
    iterable = batch_iterator(X_train, y_train, batch_size)
    if show_progress:
        iterable = tqdm(iterable, total=n_batches, desc="SGD: scaler.partial_fit")

    for _, _, X_batch, _ in iterable:
        scaler.partial_fit(X_batch.values)

    # 2) partial_fit classifier
    iterable = batch_iterator(X_train, y_train, batch_size)
    if show_progress:
        iterable = tqdm(iterable, total=n_batches, desc="SGD: clf.partial_fit")

    first_batch = True
    for _, _, X_batch, y_batch in iterable:
        Xb = scaler.transform(X_batch.values)
        if first_batch:
            clf.partial_fit(Xb, y_batch, classes=classes)
            first_batch = False
        else:
            clf.partial_fit(Xb, y_batch)

    fit_seconds = time.time() - t0
    wrapped = ScaledModelWrapper(scaler, clf)
    return wrapped, fit_seconds


def train_logreg_sampled(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    n_samples: int = 1_000_000,
    max_iter: int = 120,
    random_state: int = 42
):
    X_sub, y_sub = stratified_subsample(
        X_train,
        y_train,
        n_samples=n_samples,
        random_state=random_state
    )

    scaler = StandardScaler()
    X_sub_scaled = scaler.fit_transform(X_sub.values)

    model = LogisticRegression(
        solver="saga",
        max_iter=max_iter,
        random_state=random_state,
        verbose=0
    )

    t0 = time.time()
    model.fit(X_sub_scaled, y_sub)
    fit_seconds = time.time() - t0

    wrapped = ScaledModelWrapper(scaler, model)
    notes = f"sampled_train={len(X_sub):,}"
    return wrapped, fit_seconds, notes


def train_extratrees_sampled(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    n_samples: int = 400_000,
    n_estimators: int = 300,
    random_state: int = 42
):
    X_sub, y_sub = stratified_subsample(X_train, y_train, n_samples=n_samples, random_state=random_state)

    model = ExtraTreesClassifier(
        n_estimators=n_estimators,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=random_state,
        verbose=0
    )

    t0 = time.time()
    model.fit(X_sub, y_sub)
    fit_seconds = time.time() - t0
    notes = f"sampled_train={len(X_sub):,}"
    return model, fit_seconds, notes


def train_hgb_sampled(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    n_samples: int = 600_000,
    learning_rate: float = 0.08,
    max_iter: int = 250,
    max_leaf_nodes: int = 31,
    random_state: int = 42
):
    X_sub, y_sub = stratified_subsample(X_train, y_train, n_samples=n_samples, random_state=random_state)

    model = HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=learning_rate,
        max_iter=max_iter,
        max_leaf_nodes=max_leaf_nodes,
        min_samples_leaf=40,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
        random_state=random_state,
        verbose=0
    )

    t0 = time.time()
    model.fit(X_sub, y_sub)
    fit_seconds = time.time() - t0
    notes = f"sampled_train={len(X_sub):,}"
    return model, fit_seconds, notes


# =========================================================
# 5. Главный пайплайн
# =========================================================

def run_hvac_classic_ml_pipeline(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    train_target,
    val_target,
    test_target,
    use_datetime_features: bool = False,
    reduce_memory: bool = True,
    verbose_memory: bool = False,
    random_state: int = 42,
    # размеры подвыборок
    sample_size_lr: int = 1_000_000,
    sample_size_tree: int = 400_000,
    sample_size_hgb: int = 600_000,
    # SGD
    sgd_batch_size: int = 200_000,
    # включение моделей
    run_dummy: bool = True,
    run_sgd: bool = True,
    run_logreg: bool = True,
    run_extratrees: bool = True,
    run_hgb: bool = True,
    # печать
    show_progress: bool = True,
    show_classification_report_for_best: bool = True
) -> Dict[str, Any]:
    """
    Полный notebook-friendly пайплайн:
    - готовит X, y
    - учит несколько baseline моделей
    - сравнивает на val
    - считает test для лучшей
    - возвращает все артефакты в словаре
    """

    pipeline_start = time.time()

    # -----------------------------------------------------
    # Шаг 1. Подготовка данных
    # -----------------------------------------------------
    print_step("STEP 1/6. Prepare features")
    X_train = prepare_features_from_multiindex(
        train_df,
        use_datetime_features=use_datetime_features,
        reduce_memory=reduce_memory,
        verbose=verbose_memory
    )
    X_val = prepare_features_from_multiindex(
        val_df,
        use_datetime_features=use_datetime_features,
        reduce_memory=reduce_memory,
        verbose=verbose_memory
    )
    X_test = prepare_features_from_multiindex(
        test_df,
        use_datetime_features=use_datetime_features,
        reduce_memory=reduce_memory,
        verbose=verbose_memory
    )

    print(f"X_train shape: {X_train.shape}")
    print(f"X_val   shape: {X_val.shape}")
    print(f"X_test  shape: {X_test.shape}")

    # -----------------------------------------------------
    # Шаг 2. Подготовка target
    # -----------------------------------------------------
    print_step("STEP 2/6. Prepare targets")
    y_train, label_encoder = prepare_target(train_target, label_encoder=None)
    y_val, _ = prepare_target(val_target, label_encoder=label_encoder)
    y_test, _ = prepare_target(test_target, label_encoder=label_encoder)

    summarize_target(y_train, label_encoder=label_encoder, name="y_train")
    summarize_target(y_val, label_encoder=label_encoder, name="y_val")
    summarize_target(y_test, label_encoder=label_encoder, name="y_test")

    classes = np.arange(len(label_encoder.classes_))
    print(f"Encoded classes: {len(classes)}")

    # Базовые проверки
    assert len(X_train) == len(y_train), "X_train and y_train lengths differ"
    assert len(X_val) == len(y_val), "X_val and y_val lengths differ"
    assert len(X_test) == len(y_test), "X_test and y_test lengths differ"

    # -----------------------------------------------------
    # Шаг 3. Обучение
    # -----------------------------------------------------
    print_step("STEP 3/6. Train models")

    trained_models: Dict[str, Any] = {}
    val_predictions: Dict[str, np.ndarray] = {}
    results: List[Dict[str, Any]] = []

    # 3.1 Dummy
    if run_dummy:
        print("\n[Model] DummyClassifier")
        model, fit_seconds = train_dummy(X_train, y_train)
        res, y_pred = evaluate_model(
            model, X_val, y_val, label_encoder,
            model_name="DummyClassifier",
            fit_seconds=fit_seconds,
            notes="baseline"
        )
        trained_models["DummyClassifier"] = model
        val_predictions["DummyClassifier"] = y_pred
        results.append(asdict(res))
        gc.collect()

    # 3.2 SGD
    if run_sgd:
        print("\n[Model] SGDClassifier(log_loss) on full train via partial_fit")
        model, fit_seconds = train_sgd_incremental(
            X_train=X_train,
            y_train=y_train,
            classes=classes,
            batch_size=sgd_batch_size,
            alpha=1e-4,
            random_state=random_state,
            show_progress=show_progress
        )
        res, y_pred = evaluate_model(
            model, X_val, y_val, label_encoder,
            model_name="SGDClassifier_logloss",
            fit_seconds=fit_seconds,
            notes=f"full_train_partial_fit, batch_size={sgd_batch_size:,}"
        )
        trained_models["SGDClassifier_logloss"] = model
        val_predictions["SGDClassifier_logloss"] = y_pred
        results.append(asdict(res))
        gc.collect()

    # 3.3 Logistic Regression
    if run_logreg:
        print("\n[Model] LogisticRegression(saga) on sampled train")
        model, fit_seconds, notes = train_logreg_sampled(
            X_train=X_train,
            y_train=y_train,
            n_samples=sample_size_lr,
            max_iter=120,
            random_state=random_state
        )
        res, y_pred = evaluate_model(
            model, X_val, y_val, label_encoder,
            model_name="LogisticRegression_saga",
            fit_seconds=fit_seconds,
            notes=notes
        )
        trained_models["LogisticRegression_saga"] = model
        val_predictions["LogisticRegression_saga"] = y_pred
        results.append(asdict(res))
        gc.collect()

    # 3.4 ExtraTrees
    if run_extratrees:
        print("\n[Model] ExtraTreesClassifier on sampled train")
        model, fit_seconds, notes = train_extratrees_sampled(
            X_train=X_train,
            y_train=y_train,
            n_samples=sample_size_tree,
            n_estimators=300,
            random_state=random_state
        )
        res, y_pred = evaluate_model(
            model, X_val, y_val, label_encoder,
            model_name="ExtraTreesClassifier",
            fit_seconds=fit_seconds,
            notes=notes
        )
        trained_models["ExtraTreesClassifier"] = model
        val_predictions["ExtraTreesClassifier"] = y_pred
        results.append(asdict(res))
        gc.collect()

    # 3.5 HistGradientBoosting
    if run_hgb:
        print("\n[Model] HistGradientBoostingClassifier on sampled train")
        model, fit_seconds, notes = train_hgb_sampled(
            X_train=X_train,
            y_train=y_train,
            n_samples=sample_size_hgb,
            learning_rate=0.08,
            max_iter=250,
            max_leaf_nodes=31,
            random_state=random_state
        )
        res, y_pred = evaluate_model(
            model, X_val, y_val, label_encoder,
            model_name="HistGradientBoostingClassifier",
            fit_seconds=fit_seconds,
            notes=notes
        )
        trained_models["HistGradientBoostingClassifier"] = model
        val_predictions["HistGradientBoostingClassifier"] = y_pred
        results.append(asdict(res))
        gc.collect()

    results_df = pd.DataFrame(results).sort_values(
        ["val_macro_f1", "val_accuracy"],
        ascending=False
    ).reset_index(drop=True)

    # -----------------------------------------------------
    # Шаг 4. Leaderboard на validation
    # -----------------------------------------------------
    print_step("STEP 4/6. Validation leaderboard")
    display(results_df)

    best_model_name = results_df.iloc[0]["model_name"]
    best_model = trained_models[best_model_name]

    print(f"Best model on validation: {best_model_name}")

    # -----------------------------------------------------
    # Шаг 5. Test для лучшей модели
    # -----------------------------------------------------
    print_step("STEP 5/6. Evaluate best model on test")
    test_metrics = evaluate_on_test(
        model=best_model,
        X_test=X_test,
        y_test=y_test,
        label_encoder=label_encoder,
        model_name=best_model_name
    )

    print(f"test_accuracy    = {test_metrics['test_accuracy']:.6f}")
    print(f"test_macro_f1    = {test_metrics['test_macro_f1']:.6f}")
    print(f"test_weighted_f1 = {test_metrics['test_weighted_f1']:.6f}")
    if test_metrics["test_logloss"] is not None:
        print(f"test_logloss     = {test_metrics['test_logloss']:.6f}")

    if show_classification_report_for_best:
        print("\nClassification report:\n")
        print(test_metrics["classification_report"])

    # -----------------------------------------------------
    # Шаг 6. Feature importance / coefficients для лучшего
    # -----------------------------------------------------
    print_step("STEP 6/6. Extract best-model artifacts")

    feature_names = list(X_train.columns)
    feature_importance_df = None

    # ExtraTrees / HGB
    if hasattr(best_model, "feature_importances_"):
        feature_importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance": best_model.feature_importances_
        }).sort_values("importance", ascending=False).reset_index(drop=True)

    # Scaled wrapper with linear model
    elif isinstance(best_model, ScaledModelWrapper) and hasattr(best_model.model, "coef_"):
        coef = best_model.model.coef_  # shape: (n_classes, n_features)
        feature_importance_df = pd.DataFrame({
            "feature": feature_names,
            "mean_abs_coef": np.mean(np.abs(coef), axis=0)
        }).sort_values("mean_abs_coef", ascending=False).reset_index(drop=True)

    if feature_importance_df is not None:
        print("Top features of best model:")
        display(feature_importance_df.head(20))
    else:
        print("No feature importance available for best model.")

    total_seconds = time.time() - pipeline_start
    print(f"\nPipeline finished in {total_seconds / 60:.2f} min")

    artifacts = {
        "config": {
            "use_datetime_features": use_datetime_features,
            "reduce_memory": reduce_memory,
            "random_state": random_state,
            "sample_size_lr": sample_size_lr,
            "sample_size_tree": sample_size_tree,
            "sample_size_hgb": sample_size_hgb,
            "sgd_batch_size": sgd_batch_size
        },
        "data": {
            "X_train_shape": X_train.shape,
            "X_val_shape": X_val.shape,
            "X_test_shape": X_test.shape,
            "n_classes": len(label_encoder.classes_),
            "feature_names": feature_names
        },
        "label_encoder": label_encoder,
        "prepared": {
            "X_train": X_train,
            "X_val": X_val,
            "X_test": X_test,
            "y_train": y_train,
            "y_val": y_val,
            "y_test": y_test
        },
        "models": trained_models,
        "validation": {
            "results_df": results_df,
            "val_predictions": val_predictions,
            "best_model_name": best_model_name
        },
        "test": test_metrics,
        "best_model": best_model,
        "best_model_feature_importance": feature_importance_df,
        "timing": {
            "total_seconds": total_seconds
        }
    }

    return artifacts

In [10]:
artifacts = run_hvac_classic_ml_pipeline(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    train_target=train_target,
    val_target=val_target,
    test_target=test_target,

    use_datetime_features=False,   # сначала лучше False
    reduce_memory=True,
    verbose_memory=False,

    random_state=42,

    sample_size_lr=1_000_000,
    sample_size_tree=400_000,
    sample_size_hgb=600_000,

    sgd_batch_size=200_000,

    run_dummy=True,
    run_sgd=True,
    run_logreg=True,
    run_extratrees=True,
    run_hgb=True,

    show_progress=True,
    show_classification_report_for_best=True
)


STEP 1/6. Prepare features
X_train shape: (4448700, 25)
X_val   shape: (1317600, 25)
X_test  shape: (1899361, 25)

STEP 2/6. Prepare targets
y_train: n=4,448,700, classes=15
y_train preview counts: [(np.int64(0), np.int64(305220)), (np.int64(1), np.int64(305220)), (np.int64(2), np.int64(305220)), (np.int64(3), np.int64(305220)), (np.int64(4), np.int64(305220)), (np.int64(5), np.int64(305220)), (np.int64(6), np.int64(305220)), (np.int64(7), np.int64(305220)), (np.int64(8), np.int64(305220)), (np.int64(9), np.int64(305220))]
y_val: n=1,317,600, classes=15
y_val preview counts: [(np.int64(0), np.int64(87840)), (np.int64(1), np.int64(87840)), (np.int64(2), np.int64(87840)), (np.int64(3), np.int64(87840)), (np.int64(4), np.int64(87840)), (np.int64(5), np.int64(87840)), (np.int64(6), np.int64(87840)), (np.int64(7), np.int64(87840)), (np.int64(8), np.int64(87840)), (np.int64(9), np.int64(87840))]
y_test: n=1,899,361, classes=15
y_test preview counts: [(np.int64(0), np.int64(132480)), (np.int

SGD: scaler.partial_fit:   0%|          | 0/23 [00:00<?, ?it/s]

SGD: clf.partial_fit:   0%|          | 0/23 [00:00<?, ?it/s]

SGDClassifier_logloss:
  fit_seconds    = 16.64
  pred_seconds   = 0.28
  val_accuracy   = 0.147272
  val_macro_f1   = 0.099023
  val_weighted_f1= 0.099023
  val_logloss    = 10.411450
  notes          = full_train_partial_fit, batch_size=200,000

[Model] LogisticRegression(saga) on sampled train
LogisticRegression_saga:
  fit_seconds    = 46.34
  pred_seconds   = 0.36
  val_accuracy   = 0.632319
  val_macro_f1   = 0.610775
  val_weighted_f1= 0.610775
  val_logloss    = 1.144053
  notes          = sampled_train=1,000,000

[Model] ExtraTreesClassifier on sampled train
ExtraTreesClassifier:
  fit_seconds    = 58.78
  pred_seconds   = 36.59
  val_accuracy   = 0.865904
  val_macro_f1   = 0.865279
  val_weighted_f1= 0.865279
  val_logloss    = 0.373188
  notes          = sampled_train=400,000

[Model] HistGradientBoostingClassifier on sampled train
HistGradientBoostingClassifier:
  fit_seconds    = 112.23
  pred_seconds   = 52.56
  val_accuracy   = 0.859133
  val_macro_f1   = 0.858377
  val

,model_name,fit_seconds,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,notes
0,ExtraTreesClassifier,58.778873,0.865904,0.865279,0.865279,0.373188,"sampled_train=400,000"
1,HistGradientBoostingClassifier,112.233821,0.859133,0.858377,0.858377,0.391364,"sampled_train=600,000"
2,LogisticRegression_saga,46.337903,0.632319,0.610775,0.610775,1.144053,"sampled_train=1,000,000"
3,SGDClassifier_logloss,16.639633,0.147272,0.099023,0.099023,10.411450,"full_train_partial_fit, batch_size=200,000"
4,DummyClassifier,0.079854,0.066667,0.008333,0.008333,2.716182,baseline


Best model on validation: ExtraTreesClassifier

STEP 5/6. Evaluate best model on test
test_accuracy    = 0.794894
test_macro_f1    = 0.801206
test_weighted_f1 = 0.794338
test_logloss     = 0.534746

Classification report:

              precision    recall  f1-score   support

           0     0.4611    0.5445    0.4993    132480
           1     0.6079    0.6586    0.6322    132480
           2     0.6278    0.7727    0.6928    132480
           3     0.7068    0.6315    0.6670    132480
           4     0.7230    0.6381    0.6779    132480
           5     0.6798    0.5855    0.6291    132480
           6     0.7900    0.8459    0.8170    132480
           7     0.9956    0.9986    0.9971    132480
           8     0.9992    0.9992    0.9992    132480
           9     0.9997    0.9996    0.9997    132480
          10     0.8690    0.8879    0.8783    132480
          11     0.9998    1.0000    0.9999    132480
          12     0.9999    0.9993    0.9996    132480
          13     0.9

,feature,importance
0,CHWC_VLV,0.247176
1,SA_TEMP,0.118246
2,OA_DMPR,0.109159
3,RA_DMPR,0.107931
4,CHWC_VLV_DM,0.081813
5,MA_TEMP,0.051909
6,ZONE_TEMP_1,0.028491
7,OA_DMPR_DM,0.023966
8,RA_DMPR_DM,0.023740
9,OA_TEMP,0.021017



Pipeline finished in 7.92 min


In [11]:
from sklearn.metrics import confusion_matrix
import pandas as pd

y_val = artifacts["prepared"]["y_val"]
X_val = artifacts["prepared"]["X_val"]
model = artifacts["best_model"]
le = artifacts["label_encoder"]

y_pred = model.predict(X_val)

cm = confusion_matrix(y_val, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=le.inverse_transform(range(len(le.classes_))),
    columns=le.inverse_transform(range(len(le.classes_)))
)

cm_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,55305,2025,1257,3677,4710,4969,143,19,1,0,8103,2,2,253,7374
1,3011,70957,2974,2600,2181,33,7,8,14,1,10,0,0,30,6014
2,239,1093,85566,143,161,5,50,34,2,0,1,0,1,7,538
3,4638,2615,1694,65096,9339,30,11,7,9,2,1,0,0,99,4299
4,2861,2679,1228,4230,75005,22,7,9,4,1,3,0,0,22,1769
5,10913,20,10,308,6,55341,985,25,11,2,16311,0,0,244,3664
6,13,0,1,5,0,1376,86437,1,0,0,0,1,0,0,6
7,33,0,5,10,0,1,3,87762,2,0,0,0,0,24,0
8,5,39,13,0,0,3,0,47,87721,0,0,0,0,0,12
9,0,2,1,47,0,0,0,3,0,87787,0,0,0,0,0


In [12]:
cm_norm = cm / cm.sum(axis=1, keepdims=True)

cm_norm_df = pd.DataFrame(
    cm_norm,
    index=le.inverse_transform(range(len(le.classes_))),
    columns=le.inverse_transform(range(len(le.classes_)))
)

cm_norm_df.round(3)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0.630,0.023,0.014,0.042,0.054,0.057,0.002,0.000,0.000,0.000,0.092,0.0,0.0,0.003,0.084
1,0.034,0.808,0.034,0.030,0.025,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.068
2,0.003,0.012,0.974,0.002,0.002,0.000,0.001,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.006
3,0.053,0.030,0.019,0.741,0.106,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.001,0.049
4,0.033,0.030,0.014,0.048,0.854,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.020
5,0.124,0.000,0.000,0.004,0.000,0.630,0.011,0.000,0.000,0.000,0.186,0.0,0.0,0.003,0.042
6,0.000,0.000,0.000,0.000,0.000,0.016,0.984,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000
7,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.999,0.000,0.000,0.000,0.0,0.0,0.000,0.000
8,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.001,0.999,0.000,0.000,0.0,0.0,0.000,0.000
9,0.000,0.000,0.000,0.001,0.000,0.000,0.000,0.000,0.000,0.999,0.000,0.0,0.0,0.000,0.000
